In [2]:
# import necessary packages
import xarray as xr
import numpy as np
import pandas as pd
import os
import glob
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(rc={'axes.facecolor': 'grey'})
plt.rcParams['figure.dpi'] = 300

In [3]:
os.chdir('C:/Users/edwin/OneDrive/Documents/GitHub/CHC')

In [270]:
# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

list_of_files = glob.glob('data/csv/*.csv')
# Loop over all files
for f in list_of_files:
    # Generate the DataFrame
    df = pd.read_csv(f, sep=',', header=0, index_col=0)

    # Store the DataFrame in the dictionary with the year as key
    dfs_dict[f] = df

# Concatenate all DataFrames in the dictionary into one DataFrame
final_df = pd.concat(dfs_dict.values(), ignore_index=True)

# Convert the 'date_of_prediction' column to datetime format
final_df['date_of_prediction'] = pd.to_datetime(final_df['date_of_prediction'])
final_df['month_of_prediction'] = final_df['date_of_prediction'].dt.month
final_df = final_df.drop(columns=['date_of_prediction', 'realization_year'])
final_df['precip'] = final_df['precip']/30

In [271]:
# Calculating all the statistics for each region and model
# groupby the region, model, season and month_of_prediction and calculate the corr between the predicted and actual precipitation for future use
corr = final_df.groupby(['region', 'model', 'season', 'month_of_prediction'])[['predicted_precip', 'precip']].corr(method = 'spearman').drop(['precip'], axis = 1).reset_index()
corr = corr.drop(corr.index[::2]).drop(columns = ['level_4'])
corr = corr.rename(columns = {'predicted_precip': 'corr'})

# Calculate mean and standard deviation
stat = final_df.groupby(['region', 'model', 'season', 'month_of_prediction']).agg(['mean', 'std']).reset_index()
stat.columns = ['region', 'model', 'season', 'month_of_prediction', 'pred_mean', 'pred_std', 'actual_mean', 'actual_std']

# Merging stat and spatial_means_corr to get 1 df with all values
stat_clean = stat.merge(corr, left_on=['region', 'model', 'season', 'month_of_prediction'], right_on=['region', 'model', 'season', 'month_of_prediction'], how='left').dropna()

# Calculating metrics
stat_clean['potential_skill'] = np.square(stat_clean['corr'])
stat_clean['conditional_bias'] = np.square(stat_clean['corr'] - (stat_clean['pred_std'] / stat_clean['actual_std']))
stat_clean['unconditional_bias'] = np.square((stat_clean['pred_mean'] - stat_clean['actual_mean']) / stat_clean['actual_std'])
stat_clean['skill_score'] = stat_clean['potential_skill'] - stat_clean['conditional_bias'] - stat_clean['unconditional_bias']

# dropping unnecessary columns
stat_clean['model'].unique()
stat_clean[stat_clean['model'] == 'SMME']

,region,model,season,month_of_prediction,pred_mean,pred_std,actual_mean,actual_std,corr,potential_skill,conditional_bias,unconditional_bias,skill_score
172,eastern_east_africa,SMME,MAM,1,1.975970,0.180925,2.276974,0.562590,0.390762,0.152695,0.004784,0.286262,-0.138351
173,eastern_east_africa,SMME,MAM,2,2.051038,0.249653,2.276974,0.562590,0.423021,0.178946,0.000430,0.161283,0.017233
174,eastern_east_africa,SMME,MAM,3,2.117986,0.323942,2.276974,0.562590,0.802786,0.644465,0.051521,0.079863,0.513081
175,eastern_east_africa,SMME,MAM,9,2.145762,0.233688,2.276974,0.562590,0.507331,0.257385,0.008455,0.054396,0.194534
176,eastern_east_africa,SMME,MAM,10,1.511606,0.182865,2.261371,0.567684,0.314150,0.098690,0.000064,1.744364,-1.645737
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1382,west_africa,SMME,JAS,3,7.184475,0.386880,7.368274,0.733531,0.461144,0.212654,0.004393,0.062784,0.145476
1383,west_africa,SMME,JAS,4,7.193658,0.373034,7.368274,0.733531,0.472874,0.223610,0.001272,0.056667,0.165670
1384,west_africa,SMME,JAS,5,7.438115,0.424314,7.368274,0.733531,0.379765,0.144222,0.039477,0.009065,0.095679
1385,west_africa,SMME,JAS,6,7.559975,0.464952,7.368274,0.733531,0.454912,0.206945,0.032020,0.068298,0.106626


In [272]:
# Create a combined column for the region-season pair
stat_clean['region_season'] = stat_clean['region'] + " | " + stat_clean['season']

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='potential_skill')
    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }
    
    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]
    
    sns.heatmap(d,
                vmin=0, vmax=0.5,
                cmap=sns.color_palette('Reds', 10),
                fmt=".2f",
                linewidths=0.1, linecolor='black',
                square=True)
    plt.xticks(fontsize=7)
    plt.yticks(fontsize=7)
    plt.gca().invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(stat_clean, col='region_season', sharex=False, sharey=False, col_wrap=4)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")
fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Potential Skill by Model and Month of Prediction', y=0.98)

plt.savefig('figures/seasonal_metrics/potential_skill.png')
plt.close()

,region,model,season,month_of_prediction,corr,potential_skill,conditional_bias,unconditional_bias,skill_score,region_season
0,eastern_east_africa,CCSM4,MAM,1,0.260264,0.067737,3.587119e-03,2.125987,-2.061836,eastern_east_africa | MAM
1,eastern_east_africa,CCSM4,MAM,2,0.374633,0.140350,9.484789e-03,1.964059,-1.833193,eastern_east_africa | MAM
2,eastern_east_africa,CCSM4,MAM,3,0.614003,0.377000,1.170352e-07,1.296031,-0.919031,eastern_east_africa | MAM
3,eastern_east_africa,CCSM4,MAM,9,0.052419,0.002748,1.746199e-02,3.906009,-3.920723,eastern_east_africa | MAM
4,eastern_east_africa,CCSM4,MAM,10,-0.005499,0.000030,5.534380e-02,3.893003,-3.948317,eastern_east_africa | MAM
...,...,...,...,...,...,...,...,...,...,...
1285,west_africa,NCEP,JAS,3,0.488270,0.238407,2.443729e-01,0.119563,-0.125529,west_africa | JAS
1286,west_africa,NCEP,JAS,4,0.532991,0.284080,1.212464e-01,0.694241,-0.531408,west_africa | JAS
1287,west_africa,NCEP,JAS,5,0.394062,0.155285,2.007519e-01,2.874216,-2.919683,west_africa | JAS
1288,west_africa,NCEP,JAS,6,0.459677,0.211303,1.728182e-01,3.529483,-3.490997,west_africa | JAS


In [82]:
# keep first 3 months of prediction of each model for each region and season\
def keep_first_months_of_prediction(df):
    temp = df.copy()
    for season in temp['season'].unique():
        if (temp[temp['season'] == season]['month_of_prediction'] == 12).any() \
        & (temp[temp['season'] == season]['month_of_prediction'] == 1).any():
            temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= 8),'month_of_prediction'] = \
            temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= 8),'month_of_prediction'] - 12
        # keep 3 largest months of prediction of each model for each region and season
        max = temp[temp['season'] == season]['month_of_prediction'].max()
        temp.loc[(temp['season'] == season)] = \
        temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= (max - 2))]
    return temp.dropna()


In [122]:
potential_skill = keep_first_months_of_prediction(stat_clean[stat_clean['model'] != 'MME'])
potential_skill = potential_skill.drop(columns = ['corr', 'conditional_bias', 'unconditional_bias', 'skill_score'])

# taking the mean of the month of prediction for each region and model and season
potential_skill = potential_skill.groupby(['region', 'model', 'season'])[['potential_skill']].mean().reset_index()
potential_skill

,region,model,season,potential_skill
0,eastern_east_africa,CCSM4,MAM,0.195029
1,eastern_east_africa,CCSM4,OND,0.249536
2,eastern_east_africa,CESM1,MAM,0.241240
3,eastern_east_africa,CESM1,OND,0.370867
4,eastern_east_africa,CMCC,MAM,0.262210
...,...,...,...,...
175,west_africa,GFDL,JAS,0.104363
176,west_africa,JMA,JAS,0.032314
177,west_africa,METEO,JAS,0.014781
178,west_africa,NASA,JAS,0.158595


In [123]:
an = pd.read_csv('data/csv/metrics/region_model_df_high.csv', sep=',', header=0, index_col=0)
bn = pd.read_csv('data/csv/metrics/region_model_df_low.csv', sep=',', header=0, index_col=0)
an = keep_first_months_of_prediction(an)
bn = keep_first_months_of_prediction(bn)
an = an.rename(columns = {'agreement': 'an_agreement'})
bn = bn.rename(columns = {'agreement': 'bn_agreement'})

# taking the month of prediction mean for each region and model and season
an = an[an['model'] != 'MME'].groupby(['region', 'model', 'season'])[['an_agreement']].mean().reset_index()
bn = bn[bn['model'] != 'MME'].groupby(['region', 'model', 'season'])[['bn_agreement']].mean().reset_index()

In [ ]:
# merge an, bn and potential_skill on region, model 
an_bn = an.merge(bn, left_on=['region', 'model', 'season'], right_on=['region', 'model', 'season'], how='left')
an_bn = an_bn.dropna()

In [269]:
# merge potential_skill with an_bn on region, model and season
merged = potential_skill.merge(an_bn, left_on=['region', 'model', 'season'], right_on=['region', 'model', 'season'], how='left')
merged = merged.dropna()

# keep models with potential skill > 0.3 and an_agreement > 0.4 and bn_agreement > 0.4
merged = merged[(merged['potential_skill'] > 0.3) | ((merged['an_agreement'] > 0.4) & (merged['bn_agreement'] > 0.4))]
merged['region_season'] = merged['region'] + " | " + merged['season'] # compile region and season into one column, split by " | "
merged = merged.drop(columns = ['region', 'season'])

# create a nested dictionary where the keys are the regions, the values are lists of models
models = merged.groupby('region_season')['model'].apply(list).to_dict()
models

# separate the region and season into nested keys
SMME_models = {}
for region_season, model_list in models.items():
    region, season = region_season.split(" | ")
    if region not in SMME_models:
        SMME_models[region] = {}
    SMME_models[region][season] = model_list
SMME_models['south_sudan']
print(SMME_models)

{'eastern_east_africa': {'MAM': ['CCSM4', 'CESM1', 'CMCC', 'CanESM5', 'DWD', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NASA', 'NCEP'], 'OND': ['CCSM4', 'CESM1', 'CMCC', 'CanESM5', 'DWD', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NASA']}, 'eastern_ukraine': {'AMJ': ['GEM5'], 'DJF': ['NCEP'], 'JA': ['CMCC', 'CanESM5']}, 'lake_victoria_basin': {'DJF': ['CMCC'], 'MAM': ['CCSM4', 'CESM1', 'CMCC', 'CanESM5', 'NASA', 'NCEP'], 'SON': ['CanESM5', 'ECMWF', 'GFDL', 'METEO', 'NASA', 'NCEP']}, 'south_sudan': {'ASO': ['CMCC', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NCEP'], 'JAS': ['CESM1', 'DWD', 'ECMWF', 'GEM5', 'JMA', 'METEO', 'NCEP'], 'MJJ': ['CMCC', 'CanESM5', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'NASA']}, 'southern_africa': {'FMA': ['DWD', 'ECMWF', 'GFDL', 'JMA', 'NASA']}, 'sri_lanka': {'OND': ['CCSM4', 'CESM1', 'CMCC', 'DWD', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NASA', 'NCEP']}, 'west_africa': {'JAS': ['ECMWF', 'GFDL', 'NASA', 'NCEP']}}


In [275]:
with open('SMME_models.txt', 'w') as f:
    for region, seasons in SMME_models.items():
        f.write(f"{region}:\n")
        for season, models in seasons.items():
            f.write(f"  {season}: {models}\n")

In [267]:
# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

# initiate file list
list_of_files = glob.glob('data/csv/*.csv')

files_path = []

for path in list_of_files:
    path_mod = path.replace('\\', '/')
    files_path.append(path_mod)

for f in files_path:
    df = pd.read_csv(f)
    df['region'] = '_'.join(f.split('/')[-1].split('_')[0:-3])
    dfs_dict[f] = df

df = pd.concat(dfs_dict.values(), ignore_index=True)
smme_df = pd.DataFrame()
# keep only the rows with the models in SMME_models that matches with the keys in SMME_models
for region, seasons in SMME_models.items():
    for season, models in seasons.items():
        temp = df[(df['model'].isin(models)) & (df['region'] == region) & (df['season'] == season)]
        smme_df = pd.concat([smme_df, temp], ignore_index=True)
        
smme_df = smme_df[['region', 'season', 'date_of_prediction', 'realization_year', 'predicted_precip', 'precip']]\
                .groupby(['region', 'season', 'date_of_prediction', 'realization_year'])[['predicted_precip', 'precip']].mean().reset_index()
smme_df['model'] = 'SMME'
smme_df

,region,season,date_of_prediction,realization_year,predicted_precip,precip,model
0,eastern_east_africa,MAM,1992-09-01,1993,2.014308,62.90869,SMME
1,eastern_east_africa,MAM,1992-10-01,1993,1.908853,62.90869,SMME
2,eastern_east_africa,MAM,1992-11-01,1993,2.013665,62.90869,SMME
3,eastern_east_africa,MAM,1992-12-01,1993,2.179031,62.90869,SMME
4,eastern_east_africa,MAM,1993-01-01,1993,2.162677,62.90869,SMME
...,...,...,...,...,...,...,...
3411,west_africa,JAS,2024-03-01,2024,8.106143,255.36578,SMME
3412,west_africa,JAS,2024-04-01,2024,8.189667,255.36578,SMME
3413,west_africa,JAS,2024-05-01,2024,8.145302,255.36578,SMME
3414,west_africa,JAS,2024-06-01,2024,8.662673,255.36578,SMME


In [268]:
grouped = smme_df.groupby('region')
for region, region_df in grouped:
    region_name = str(region)
    model = 'SMME'
    filename = f'data/csv/{region_name}_{model}_merged_seasonal.csv'
    region_df.to_csv(filename)

In [274]:
# Create a combined column for the region-season pair
stat_clean['region_season'] = stat_clean['region'] + " | " + stat_clean['season']

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='conditional_bias')

    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }

    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]

    sns.heatmap(d,
                vmin=0, vmax=1,
                cmap=sns.color_palette('Blues', 10),
                fmt=".2f",
                linewidths=0.1, linecolor='black',
                square=True)
    plt.xticks(fontsize=7)
    plt.yticks(fontsize=7)
    plt.gca().invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(stat_clean, col='region_season', sharex=False, sharey=False, col_wrap=4)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")
fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Conditional Bias by Model and Month of Prediction', y=0.98)

plt.savefig('figures/seasonal_metrics/conditional_bias.png')
plt.close()

In [273]:
# Create a combined column for the region-season pair
stat_clean['region_season'] = stat_clean['region'] + " | " + stat_clean['season']

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='unconditional_bias')

    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }

    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]

    sns.heatmap(d,
                cmap=sns.color_palette('Blues', 10),
                fmt=".2f",
                linewidths=0.1, linecolor='black',
                square=True)
    plt.xticks(fontsize=7)
    plt.yticks(fontsize=7)
    plt.gca().invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(stat_clean, col='region_season', sharex=False, sharey=False, col_wrap=4)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")
fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Unconditional Bias by Model and Month of Prediction', y=0.98)

plt.savefig('figures/seasonal_metrics/unconditional_bias.png')
plt.close()